# FactIndic 01 – Data collection and preprocessing

Loads XL-Sum Hindi and Bengali (BBC article–summary pairs), cleans the articles (BBC boilerplate, Unicode, digits), filters by length and script, removes exact and near-duplicates (MinHash, 5-gram Jaccard ≥ 0.8) between test and training articles, and samples the test, dev and training articles. It also collects a small set of naturally code-mixed news (RSS, best effort) and any documents you add yourself. XL-Sum's official splits are kept (the public mT5/IndicBART checkpoints were trained on its training split); collected documents are split 70/10/20 by article.

| Setting (right-hand panel) | Value |
|---|---|
| Accelerator | **None (CPU)** |
| Internet | **On** |
| Inputs to attach | the `factindic-code` dataset; optionally a dataset with your own code-mixed documents (a CSV/JSONL whose file name contains `natural`, columns `lang`, `text`) |
| Time (full profile) | 10–25 min |

Run it with **Save Version → Save & Run All (Commit)**. When it finishes, its output (folder `factindic_out`) is
ready to attach to the next notebook. Each notebook's output contains every earlier stage, so you only ever attach
the previous notebook.

In [1]:
# ── Setup: make the factindic code importable ─────────────────────────────────────────
# Kaggle: upload factindic-code.zip as a private dataset and attach it
#         (right-hand panel -> Add Input -> Your Work / Datasets -> factindic-code -> "+").
# Colab:  put factindic-code.zip in My Drive (and run the Google Drive cell first), or upload it to /content.
# Or:     push the repository to GitHub and paste its URL here (needs Internet).
REPO_URL = ""   # e.g. "https://github.com/<your-username>/factindic"

import glob, os, shutil, subprocess, sys, zipfile

ON_COLAB = "google.colab" in sys.modules
SRC = "/content/factindic_src" if ON_COLAB else "/kaggle/working/factindic_src"

def _is_pkg(d):
    return all(os.path.exists(os.path.join(d, f)) for f in ("__init__.py", "pipeline.py", "config.py"))

def _candidates():
    inits = glob.glob("/kaggle/input/**/__init__.py", recursive=True)
    zips = glob.glob("/kaggle/input/**/*.zip", recursive=True)
    if ON_COLAB:  # shallow search only: Google Drive can be huge
        for base in ("/content", "/content/drive/MyDrive"):
            for depth in ("", "*/", "*/*/"):
                inits += glob.glob(f"{base}/{depth}factindic/__init__.py")
            zips += [z for z in glob.glob(f"{base}/*.zip") + glob.glob(f"{base}/*/*.zip")
                     if "factindic" in os.path.basename(z).lower()]
    return sorted(set(inits)), sorted(set(zips))

def _unzip(z):
    try:
        with zipfile.ZipFile(z) as zf:
            hits = [n for n in zf.namelist() if n.endswith("factindic/__init__.py")]
            if hits:
                shutil.rmtree(SRC, ignore_errors=True)
                zf.extractall(SRC)
                return os.path.dirname(os.path.dirname(os.path.join(SRC, sorted(hits, key=len)[-1])))
    except zipfile.BadZipFile:
        pass
    return None

def _find_code():
    inits, zips = _candidates()
    # 0) Colab: always unpack the newest zip from Drive, so replacing the zip there updates the code
    if ON_COLAB:
        for z in sorted(zips, key=os.path.getmtime, reverse=True):
            code = _unzip(z)
            if code:
                print("using", z)
                return code
    # 1) the repository folder (…/factindic/factindic/__init__.py)
    for p in inits:
        d = os.path.dirname(p)
        if _is_pkg(d) and os.path.basename(d) == "factindic":
            return os.path.dirname(d)
    # 2) only the inner package was uploaded (files directly in the dataset): copy it under the right name
    for p in inits:
        d = os.path.dirname(p)
        if _is_pkg(d):
            shutil.copytree(d, os.path.join(SRC, "factindic"), dirs_exist_ok=True)
            for req in glob.glob(os.path.join(os.path.dirname(d), "**", "requirements.txt"), recursive=True)[:1]:
                shutil.copy(req, SRC)
            return SRC
    # 3) a zip that nobody unpacked
    for z in zips:
        code = _unzip(z)
        if code:
            return code
    # 4) running locally from the repository
    for p in ("factindic", "../factindic"):
        if _is_pkg(p):
            return os.path.abspath(os.path.dirname(p) or ".")
    return None

CODE = _find_code()
if CODE is None and REPO_URL:
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, SRC], check=True)
    CODE = SRC
if CODE is None:
    where = (sorted(glob.glob("/content/*")) + sorted(glob.glob("/content/drive/MyDrive/*"))[:30]) if ON_COLAB else \
            (sorted(glob.glob("/kaggle/input/*")) + sorted(glob.glob("/kaggle/input/*/*"))[:30])
    how = ("Put factindic-code.zip in My Drive and run the Google Drive cell first (or upload the zip to /content)."
           if ON_COLAB else
           "Attach the dataset: right-hand panel -> Add Input -> Your Work (or Datasets) -> factindic-code -> '+'. "
           "If you uploaded it a minute ago, wait until Kaggle finishes processing it.")
    raise FileNotFoundError("factindic code not found.\n" + how + "\nWhat this notebook can see:\n  "
                            + ("\n  ".join(where) if where else "(nothing)"))
sys.path.insert(0, CODE)
os.environ["PYTHONPATH"] = CODE + os.pathsep + os.environ.get("PYTHONPATH", "")
req = os.path.join(CODE, "requirements.txt")
pkgs = ["-r", req] if os.path.exists(req) else ["indic-transliteration", "wordfreq", "datasketch", "bert-score",
                                                "python-docx", "feedparser", "trafilatura", "peft", "openpyxl",
                                                "bitsandbytes"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
import factindic
from factindic import Config
from factindic import pipeline as P
print("factindic", factindic.__version__, "loaded from", CODE)

factindic 1.0.0 loaded from /kaggle/input/datasets/lkr1196/factindic-code/factindic


In [2]:
# ── Configuration ────────────────────────────────────────────────────────────────────
# Use the SAME profile in every notebook of the chain.
#   "full"  – the real experiment
#   "quick" – ~25 articles per language, to check the whole chain on Kaggle first
#   "mock"  – CPU dry run with stand-in models (no GPU, no downloads); never report its numbers
PROFILE = "full"
cfg = Config.make(PROFILE)
# cfg.scrape = False   # turn off RSS scraping
print(cfg.to_json())

{
  "profile": "full",
  "seed": 13,
  "langs": [
    "hi",
    "bn"
  ],
  "variants": [
    "mono",
    "cm",
    "roman"
  ],
  "data_source": "xlsum",
  "n_test": 200,
  "n_dev": 30,
  "n_train_cm": 150,
  "min_words": 120,
  "max_words_raw": 1200,
  "lead_words": 300,
  "max_sents": 14,
  "min_summary_words": 8,
  "max_summary_words": 60,
  "near_dup_threshold": 0.8,
  "natural_split": [
    0.7,
    0.1,
    0.2
  ],
  "scrape": true,
  "n_natural_per_lang": 100,
  "backend": "auto",
  "summarizer_model": "Qwen/Qwen3-4B-Instruct-2507",
  "codemix_model": "Qwen/Qwen3-8B",
  "openai_base_url": "",
  "openai_model": "",
  "openai_concurrency": 4,
  "max_model_len": 6144,
  "max_batch_tokens": 24000,
  "max_batch_size": 24,
  "repetition_penalty": 1.0,
  "workers": -1,
  "quantize": "auto",
  "summary_max_words": 40,
  "tracks": [
    "A",
    "B",
    "BV"
  ],
  "max_new_tokens_A": 160,
  "max_new_tokens_B": 512,
  "seq2seq_models": [
    "mt5",
    "indicbart"
  ],
  "mt5_model": 

In [3]:
docs = P.run_stage01(cfg)

== hi: loading XL-Sum


hindi/test/0000.parquet: reconstructing file:   0%|          |  0.00B / 21.5MB            

hindi/test/0000.parquet: downloading bytes:           |  0.00B            

  XL-Sum hindi/test: 8,847 articles from the Parquet branch


hindi/validation/0000.parquet: reconstructing file:   0%|          |  0.00B / 21.4MB            

hindi/validation/0000.parquet: downloading bytes:           |  0.00B            

... (12 outputs left out)

bengali/train/0000.parquet: reconstructing file:   0%|          |  0.00B / 33.4MB            

bengali/train/0000.parquet: downloading bytes:           |  0.00B            

  XL-Sum bengali/train: 8,102 articles from the Parquet branch
   test: 1012 raw -> 1002 after filters
   validation: 1012 raw -> 1003 after filters
   train: 8102 raw -> 7276 after filters
   selected: 200 test, 30 dev, 150 train
== scraping recent news (best effort)
  hi: 35 articles from https://feeds.bbci.co.uk/hindi/rss.xml
  hi: 0 articles from https://navbharattimes.indiatimes.com/rssfeedsdefault.cms
  hi: 11 articles from https://www.amarujala.com/rss/breaking-news.xml
  hi: 26 articles from https://www.aajtak.in/rssfeeds/?id=home
  hi: 0 articles from https://feeds.feedburner.com/ndtvkhabar-latest
  bn: 13 articles from https://feeds.bbci.co.uk/bengali/rss.xml
  bn: 0 articles from https://eisamay.com/rssfeedsdefault.cms
  bn: 28 articles from https://www.prothomalo.com/feed/
  bn: 0 articles from https://bengali.news18.com/rss/khabar/west-bengal.xml
  bn: 20 articles from https://www.sangbadpratidin.in/feed/
== natural code-mixed docs: {'bn': 61, 'hi': 72}


,lang,role,docs,words,sents,cmi,ref_words
0,bn,dev,30,216.700000,13.800000,0.000000,20.566667
1,bn,natural,61,173.901639,13.934426,0.980859,NaN
2,bn,test,200,199.205000,13.585000,0.031114,21.640000
3,bn,train,150,204.660000,13.606667,0.022572,23.173333
4,hi,dev,30,246.466667,11.900000,0.000000,25.300000
5,hi,natural,72,219.458333,13.361111,1.999742,NaN
6,hi,test,200,244.130000,11.825000,0.016855,26.570000
7,hi,train,150,248.693333,12.333333,0.068693,27.520000


In [4]:
from IPython.display import Markdown, display
display(Markdown(open(f"{cfg.out_root}/01_data/data_report.md", encoding="utf-8").read()))

# Data

## Filtering

| lang | split | raw | length_filter | summary_filter | script_filter | exact_dup | kept | near_dup_removed |
|---|---|---|---|---|---|---|---|---|
| hi | test | 8847 | 133 | 0 | 3 | 0 | 8711 | 1 |
| hi | validation | 8847 | 152 | 1 | 2 | 0 | 8692 |  |
| hi | train | 70778 | 8285 | 943 | 33 | 9 | 61508 |  |
| bn | test | 1012 | 7 | 2 | 1 | 0 | 1002 | 0 |
| bn | validation | 1012 | 6 | 3 | 0 | 0 | 1003 |  |
| bn | train | 8102 | 709 | 109 | 8 | 0 | 7276 |  |

## Selected documents

| lang | role | docs | words | sents | cmi | ref_words |
|---|---|---|---|---|---|---|
| bn | dev | 30 | 216.70 | 13.80 | 0.00 | 20.57 |
| bn | natural | 61 | 173.90 | 13.93 | 0.98 |  |
| bn | test | 200 | 199.21 | 13.59 | 0.03 | 21.64 |
| bn | train | 150 | 204.66 | 13.61 | 0.02 | 23.17 |
| hi | dev | 30 | 246.47 | 11.90 | 0.00 | 25.30 |
| hi | natural | 72 | 219.46 | 13.36 | 2.00 |  |
| hi | test | 200 | 244.13 | 11.82 | 0.02 | 26.57 |
| hi | train | 150 | 248.69 | 12.33 | 0.07 | 27.52 |



A few selected articles and their reference summaries:

In [5]:
docs[docs.role == "test"].groupby("lang").head(2)[["doc_id", "lang", "n_words", "text", "summary_ref"]]

,doc_id,lang,n_words,text,summary_ref
0,hi-test-031102_indo_burma,hi,141,दिवंगत प्रधानमंत्री राजीव गाँधी 1987 में बर्मा...,भारत के उपराष्ट्रपति भैरोसिंह शेखावत पिछले सोल...
1,hi-test-031229_kitnepakistan_review,hi,249,'कितने पाकिस्तान' उपन्यास एक बड़ा और नए ढंग का...,भारतीय साहित्य अकादमी ने वर्ष 2003 के लिए कमले...
380,bn-test-120724_humayun,bn,235,"হুমায়ুন আহমেদ বাংলাদেশ সময় দুপুর দুইটায়, নু...",বাংলাদেশের জনপ্রিয় কথাসাহিত্যিক হুমায়ূন আহমে...
381,bn-test-140813_sr_broken_family,bn,175,প্রতীকী ছবি ফলে ভেঙে যাওয়া এইসব পরিবারগুলো সম...,বাংলাদেশে একটি বড় অংশের মধ্যে বিবাহ বিচ্ছেদের...


In [6]:
import os
for root, dirs, files in os.walk(cfg.out_root):
    dirs[:] = [d for d in dirs if not d.startswith("_")]
    depth = root[len(cfg.out_root):].count(os.sep)
    if depth <= 1:
        print("  " * depth + os.path.basename(root) + "/", ", ".join(sorted(files))[:300])

factindic_out/ 
  01_data/ config.json, data_report.md, data_stats.csv, docs.jsonl
